# Optuna-подбор гиперпараметров ALS и EASE

In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd

try:
    import optuna
except ImportError as exc:
    raise ImportError("Для запуска ноутбука установите Optuna: %pip install optuna") from exc

from collaborative_experiments import (
    load_clients,
    load_transactions,
    clean_transactions,
    temporal_split,
    build_all_features,
    ALSRecommender,
    EASERecommender,
)

optuna.logging.set_verbosity(optuna.logging.WARNING)

RANDOM_STATE = 42
SPLIT_DATE = "2025-05-15"
INNER_SPLIT_DATE = "2025-04-15"

ALPHA_FREQ = 0.5
ALPHA_VOL = 0.5
MIN_RELEVANCE_SHARE = 0.10
MIN_TX_FOR_EVAL = 3

COLD_TX_THRESHOLD = 10
HOT_DELTA_SHARE = 0.25
NEW_SUPPLIER_TEST = 0.10
NEW_SUPPLIER_TRAIN = 0.03

N_TRIALS_ALS = 30
N_TRIALS_EASE = 30

## Вспомогательные функции

Функции ниже повторяют логику V3: ground truth строится по тестовому периоду, потенциальные клиенты определяются через изменение wallet share, а метрики считаются по ранжированному списку поставщиков.

In [2]:
def build_ground_truth_from_test(tx_test, alpha_vol=0.5, alpha_freq=0.5, min_share=0.10, min_tx=3):
    tx_per_client = tx_test.groupby("Код клиента").size()
    eligible = tx_per_client[tx_per_client >= min_tx].index
    tx_filtered = tx_test[tx_test["Код клиента"].isin(eligible)].copy()

    if "Поставщик" not in tx_filtered.columns:
        tx_filtered["Поставщик"] = tx_filtered["Тип карты"].str.extract(r"(Поставщик\d+)")

    agg = tx_filtered.groupby(["Код клиента", "Поставщик"]).agg(
        volume=("Объем", "sum"), txn=("Объем", "count")
    ).reset_index()

    tot_vol = agg.groupby("Код клиента")["volume"].transform("sum")
    tot_txn = agg.groupby("Код клиента")["txn"].transform("sum")
    agg["vol_share"] = agg["volume"] / tot_vol
    agg["txn_share"] = agg["txn"] / tot_txn
    agg["relevance"] = alpha_vol * agg["vol_share"] + alpha_freq * agg["txn_share"]
    agg = agg[agg["relevance"] >= min_share].copy()
    agg = agg.sort_values(["Код клиента", "relevance"], ascending=[True, False])

    gt = agg.groupby("Код клиента").apply(
        lambda x: pd.Series({
            "true_suppliers": x["Поставщик"].tolist(),
            "relevance_dict": dict(zip(x["Поставщик"], x["relevance"])),
            "n_suppliers": len(x),
        })
    ).reset_index()
    return gt


def compute_wallet_share_matrix(tx_df):
    if "Поставщик" not in tx_df.columns:
        tx_df = tx_df.copy()
        tx_df["Поставщик"] = tx_df["Тип карты"].str.extract(r"(Поставщик\d+)")
    vol = tx_df.groupby(["Код клиента", "Поставщик"])["Объем"].sum().unstack(fill_value=0)
    return vol.div(vol.sum(axis=1), axis=0).fillna(0)


def build_segments(tx_train_part, tx_test_part_clean, gt_part):
    share_train = compute_wallet_share_matrix(tx_train_part)
    share_test = compute_wallet_share_matrix(tx_test_part_clean)
    all_suppliers = sorted(set(share_train.columns) | set(share_test.columns))
    share_train = share_train.reindex(columns=all_suppliers, fill_value=0)
    share_test = share_test.reindex(columns=all_suppliers, fill_value=0)

    common_clients = share_train.index.intersection(share_test.index)
    share_train_c = share_train.loc[common_clients]
    share_test_c = share_test.loc[common_clients]
    delta_share = share_test_c - share_train_c

    potential_flags = pd.DataFrame(index=common_clients)
    potential_flags["new_supplier"] = ((share_train_c < NEW_SUPPLIER_TRAIN) & (share_test_c >= NEW_SUPPLIER_TEST)).any(axis=1)
    potential_flags["lost_supplier"] = ((share_train_c >= NEW_SUPPLIER_TEST) & (share_test_c < NEW_SUPPLIER_TRAIN)).any(axis=1)
    potential_flags["top1_change"] = (share_train_c.idxmax(axis=1) != share_test_c.idxmax(axis=1)) & (share_test_c.max(axis=1) > 0)
    potential_flags["big_shift"] = delta_share.abs().max(axis=1) >= HOT_DELTA_SHARE
    potential_flags["is_potential"] = potential_flags.any(axis=1)

    tx_per_client_train = tx_train_part.groupby("Код клиента").size()
    cold_clients_set = set(tx_per_client_train[tx_per_client_train < COLD_TX_THRESHOLD].index)
    potential_warm_set = set(potential_flags[potential_flags["is_potential"]].index)
    all_potential_set = potential_warm_set | cold_clients_set
    inertial_set = set(common_clients) - all_potential_set

    segments = {
        "All": gt_part.reset_index(drop=True),
        "Potential": gt_part[gt_part["Код клиента"].isin(all_potential_set)].reset_index(drop=True),
        "Cold-start": gt_part[gt_part["Код клиента"].isin(cold_clients_set)].reset_index(drop=True),
        "Hot+": gt_part[gt_part["Код клиента"].isin(potential_warm_set)].reset_index(drop=True),
        "Inertial": gt_part[gt_part["Код клиента"].isin(inertial_set)].reset_index(drop=True),
    }
    return segments, potential_flags


def _prepare_topk(predictions, k):
    df = predictions.sort_values(["Код клиента", "score"], ascending=[True, False]).copy()
    df["rank"] = df.groupby("Код клиента").cumcount() + 1
    return df[df["rank"] <= k]


def hit_rate_at_k(predictions, ground_truth, k=1):
    topk = _prepare_topk(predictions, k)
    gt_dict = ground_truth.set_index("Код клиента")["true_suppliers"].to_dict()
    hits = []
    for cid, group in topk.groupby("Код клиента"):
        if cid in gt_dict:
            hits.append(len(set(group["Поставщик"].values) & set(gt_dict[cid])) > 0)
    return float(np.mean(hits)) if hits else 0.0


def precision_at_k(predictions, ground_truth, k=2):
    topk = _prepare_topk(predictions, k)
    gt_dict = ground_truth.set_index("Код клиента")["true_suppliers"].to_dict()
    values = []
    for cid, group in topk.groupby("Код клиента"):
        if cid in gt_dict:
            values.append(len(set(group["Поставщик"].values) & set(gt_dict[cid])) / k)
    return float(np.mean(values)) if values else 0.0


def recall_at_k(predictions, ground_truth, k=2):
    topk = _prepare_topk(predictions, k)
    gt_dict = ground_truth.set_index("Код клиента")["true_suppliers"].to_dict()
    values = []
    for cid, group in topk.groupby("Код клиента"):
        if cid in gt_dict:
            true_set = set(gt_dict[cid])
            if true_set:
                values.append(len(set(group["Поставщик"].values) & true_set) / len(true_set))
    return float(np.mean(values)) if values else 0.0


def ndcg_at_k_graded(predictions, ground_truth, k=3):
    topk = _prepare_topk(predictions, k)
    gt_rel = ground_truth.set_index("Код клиента")["relevance_dict"].to_dict()
    values = []
    for cid, group in topk.groupby("Код клиента"):
        if cid not in gt_rel:
            continue
        rel_dict = gt_rel[cid]
        rel = np.array([rel_dict.get(p, 0.0) for p in group["Поставщик"].values])
        dcg = (rel / np.log2(np.arange(2, len(rel) + 2))).sum()
        ideal_rel = np.sort(list(rel_dict.values()))[::-1][:k]
        idcg = (ideal_rel / np.log2(np.arange(2, len(ideal_rel) + 2))).sum()
        values.append(dcg / idcg if idcg > 0 else 0.0)
    return float(np.mean(values)) if values else 0.0


def map_at_k(predictions, ground_truth, k=3):
    topk = _prepare_topk(predictions, k)
    gt_dict = ground_truth.set_index("Код клиента")["true_suppliers"].to_dict()
    values = []
    for cid, group in topk.groupby("Код клиента"):
        if cid not in gt_dict:
            continue
        true_list = gt_dict[cid]
        hits, sum_precs = 0, 0.0
        for i, supplier in enumerate(group["Поставщик"].values):
            if supplier in true_list:
                hits += 1
                sum_precs += hits / (i + 1.0)
        values.append(sum_precs / min(len(true_list), k) if true_list else 0.0)
    return float(np.mean(values)) if values else 0.0


def evaluate_model(predictions, gt, segment_name, model_name):
    if len(gt) == 0:
        return None
    seg_clients = set(gt["Код клиента"])
    pred_seg = predictions[predictions["Код клиента"].isin(seg_clients)]
    return {
        "Сегмент": segment_name,
        "N клиентов": len(seg_clients),
        "Модель": model_name,
        "HR@1": round(hit_rate_at_k(pred_seg, gt, k=1), 3),
        "P@2": round(precision_at_k(pred_seg, gt, k=2), 3),
        "R@2": round(recall_at_k(pred_seg, gt, k=2), 3),
        "NDCG@3": round(ndcg_at_k_graded(pred_seg, gt, k=3), 3),
        "MAP@3": round(map_at_k(pred_seg, gt, k=3), 3),
    }


def evaluate_predictions_by_segments(predictions_by_model, segments):
    rows = []
    for segment_name, gt_segment in segments.items():
        for model_name, predictions in predictions_by_model.items():
            row = evaluate_model(predictions, gt_segment, segment_name, model_name)
            if row is not None:
                rows.append(row)
    return pd.DataFrame(rows).sort_values(["Сегмент", "NDCG@3"], ascending=[True, False]).reset_index(drop=True)

## Калиброванный вариант EASE

In [3]:
class EASECalibratedRecommender:
    def __init__(self, reg=250.0, score_direction="auto"):
        self.reg = reg
        self.score_direction = score_direction
        self.base = EASERecommender(reg=reg)
        self.invert = False

    def fit(self, interaction_features):
        self.base.fit(interaction_features)
        if self.score_direction == "inverted":
            self.invert = True
        elif self.score_direction == "raw":
            self.invert = False
        else:
            sample = interaction_features["Код клиента"].drop_duplicates().head(100).tolist()
            preds = self.base.predict_scores(sample)
            ease_top = (
                preds.sort_values(["Код клиента", "score"], ascending=[True, False])
                .groupby("Код клиента")
                .head(1)
            )
            actual_top = (
                interaction_features.sort_values(["Код клиента", "volume"], ascending=[True, False])
                .groupby("Код клиента")
                .head(1)
            )
            merged = ease_top.merge(actual_top, on="Код клиента", suffixes=("_ease", "_actual"))
            agreement = (merged["Поставщик_ease"] == merged["Поставщик_actual"]).mean()
            self.invert = bool(agreement < 0.3)
        return self

    def predict_scores(self, client_ids):
        preds = self.base.predict_scores(client_ids).copy()
        if self.invert:
            preds["score"] = -preds["score"]
        return preds

## Загрузка данных и внешний test-сплит

In [4]:
clients = load_clients()
tx = load_transactions()

if "Поставщик" not in tx.columns:
    tx["Поставщик"] = tx["Тип карты"].str.extract(r"(Поставщик\d+)")

tx_train, tx_test = temporal_split(tx, SPLIT_DATE)
tx_test_clean = clean_transactions(tx_test)

print(f"Outer train: {len(tx_train):,} транзакций")
print(f"Outer test:  {len(tx_test_clean):,} транзакций после очистки")

Outer train: 574,495 транзакций
Outer test:  214,409 транзакций после очистки


In [5]:
print("Построение признаков на полном outer train...")
tf = build_all_features(tx_train, clients)

gt = build_ground_truth_from_test(
    tx_test_clean,
    alpha_vol=ALPHA_VOL,
    alpha_freq=ALPHA_FREQ,
    min_share=MIN_RELEVANCE_SHARE,
    min_tx=MIN_TX_FOR_EVAL,
)
train_clients_set = set(tf["client_profile"]["Код клиента"])
gt = gt[gt["Код клиента"].isin(train_clients_set)].reset_index(drop=True)

segments, potential_flags = build_segments(tx_train, tx_test_clean, gt)

segment_sizes = pd.DataFrame([
    {"Сегмент": name, "N клиентов": len(gt_segment)}
    for name, gt_segment in segments.items()
])
display(segment_sizes)
print(f"Клиентов в финальном ground truth: {len(gt)}")

Построение признаков на полном outer train...


,Сегмент,N клиентов
0,All,3536
1,Potential,402
2,Cold-start,166
3,Hot+,244
4,Inertial,3134


Клиентов в финальном ground truth: 3536


## Внутренний validation-сплит для Optuna

In [6]:
tx_opt_train, tx_opt_valid = temporal_split(tx_train, INNER_SPLIT_DATE)
tx_opt_valid_clean = clean_transactions(tx_opt_valid)

print(f"Optuna train: {len(tx_opt_train):,} транзакций")
print(f"Optuna valid: {len(tx_opt_valid_clean):,} транзакций после очистки")

print("Построение признаков для Optuna train...")
tf_opt = build_all_features(tx_opt_train, clients)

gt_opt = build_ground_truth_from_test(
    tx_opt_valid_clean,
    alpha_vol=ALPHA_VOL,
    alpha_freq=ALPHA_FREQ,
    min_share=MIN_RELEVANCE_SHARE,
    min_tx=MIN_TX_FOR_EVAL,
)
opt_train_clients_set = set(tf_opt["client_profile"]["Код клиента"])
gt_opt = gt_opt[gt_opt["Код клиента"].isin(opt_train_clients_set)].reset_index(drop=True)

opt_segments, opt_potential_flags = build_segments(tx_opt_train, tx_opt_valid_clean, gt_opt)
gt_opt_objective = opt_segments["Potential"] if len(opt_segments["Potential"]) > 0 else opt_segments["All"]
valid_client_ids = gt_opt_objective["Код клиента"].tolist()

display(pd.DataFrame([{ "Сегмент": name, "N клиентов": len(value) } for name, value in opt_segments.items()]))
print(f"Optuna objective segment size: {len(gt_opt_objective)}")

Optuna train: 447,317 транзакций
Optuna valid: 127,177 транзакций после очистки
Построение признаков для Optuna train...


,Сегмент,N клиентов
0,All,3371
1,Potential,420
2,Cold-start,177
3,Hot+,256
4,Inertial,2951


Optuna objective segment size: 420


## Подбор гиперпараметров ALS

In [7]:
def objective_als(trial):
    params = {
        "n_factors": trial.suggest_categorical("n_factors", [2, 3, 4, 6, 8, 12, 16, 24]),
        "n_iters": trial.suggest_categorical("n_iters", [5, 10, 15, 20, 30]),
        "reg": trial.suggest_float("reg", 1e-3, 10.0, log=True),
        "alpha": trial.suggest_float("alpha", 1.0, 80.0, log=True),
    }
    model = ALSRecommender(**params).fit(tf_opt["interaction_features"])
    preds = model.predict_scores(valid_client_ids)
    return ndcg_at_k_graded(preds, gt_opt_objective, k=3)


study_als = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
    study_name="ALS_NDCG3",
)
study_als.optimize(objective_als, n_trials=N_TRIALS_ALS, show_progress_bar=True)

print("Best ALS NDCG@3:", round(study_als.best_value, 4))
print("Best ALS params:")
display(pd.DataFrame([study_als.best_params]))

  0%|          | 0/30 [00:00<?, ?it/s]

Best ALS NDCG@3: 0.9467
Best ALS params:


,n_factors,n_iters,reg,alpha
0,4,10,0.694058,17.313028


## Подбор гиперпараметров EASE

In [8]:
def objective_ease(trial):
    params = {
        "reg": trial.suggest_float("reg", 1.0, 5000.0, log=True),
        "score_direction": trial.suggest_categorical("score_direction", ["auto", "raw", "inverted"]),
    }
    model = EASECalibratedRecommender(**params).fit(tf_opt["interaction_features"])
    preds = model.predict_scores(valid_client_ids)
    return ndcg_at_k_graded(preds, gt_opt_objective, k=3)


study_ease = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
    study_name="EASE_NDCG3",
)
study_ease.optimize(objective_ease, n_trials=N_TRIALS_EASE, show_progress_bar=True)

print("Best EASE NDCG@3:", round(study_ease.best_value, 4))
print("Best EASE params:")
display(pd.DataFrame([study_ease.best_params]))

  0%|          | 0/30 [00:00<?, ?it/s]

Best EASE NDCG@3: 0.915
Best EASE params:


,reg,score_direction
0,3.776663,inverted


## История Optuna и лучшие параметры

In [9]:
best_params_summary = pd.DataFrame([
    {
        "Модель": "ALS",
        "Validation NDCG@3": study_als.best_value,
        **study_als.best_params,
    },
    {
        "Модель": "EASE",
        "Validation NDCG@3": study_ease.best_value,
        **study_ease.best_params,
    },
])
display(best_params_summary)

als_trials = study_als.trials_dataframe().sort_values("value", ascending=False)
ease_trials = study_ease.trials_dataframe().sort_values("value", ascending=False)

print("Top-10 ALS trials")
display(als_trials.head(10))
print("Top-10 EASE trials")
display(ease_trials.head(10))

,Модель,Validation NDCG@3,n_factors,n_iters,reg,alpha,score_direction
0,ALS,0.946687,4.0,10.0,0.694058,17.313028,NaN
1,EASE,0.914992,NaN,NaN,3.776663,NaN,inverted


Top-10 ALS trials


,number,value,datetime_start,datetime_complete,duration,params_alpha,params_n_factors,params_n_iters,params_reg,state
21,21,0.946687,2026-05-09 16:37:05.548440,2026-05-09 16:37:13.873938,0 days 00:00:08.325498,17.313028,4,10,0.694058,COMPLETE
13,13,0.946541,2026-05-09 16:34:01.770146,2026-05-09 16:34:26.693098,0 days 00:00:24.922952,27.299859,4,30,0.509859,COMPLETE
12,12,0.945219,2026-05-09 16:33:38.443750,2026-05-09 16:34:01.769146,0 days 00:00:23.325396,22.534988,4,30,0.699162,COMPLETE
11,11,0.945084,2026-05-09 16:33:30.839395,2026-05-09 16:33:38.441750,0 days 00:00:07.602355,21.012585,4,10,0.586092,COMPLETE
10,10,0.945077,2026-05-09 16:33:23.327241,2026-05-09 16:33:30.838396,0 days 00:00:07.511155,21.020957,4,10,0.608795,COMPLETE
22,22,0.944819,2026-05-09 16:37:13.874943,2026-05-09 16:37:38.923560,0 days 00:00:25.048617,15.300455,4,30,1.889624,COMPLETE
26,26,0.944750,2026-05-09 16:38:16.295928,2026-05-09 16:38:23.898942,0 days 00:00:07.603014,6.652629,4,10,0.462659,COMPLETE
4,4,0.944254,2026-05-09 16:32:09.419868,2026-05-09 16:32:17.133865,0 days 00:00:07.713997,22.141811,4,10,1.827451,COMPLETE
17,17,0.943936,2026-05-09 16:35:44.070834,2026-05-09 16:36:09.087508,0 days 00:00:25.016674,13.554814,4,30,9.941451,COMPLETE
5,5,0.942329,2026-05-09 16:32:17.134869,2026-05-09 16:32:32.928916,0 days 00:00:15.794047,7.918948,12,20,3.538759,COMPLETE


Top-10 EASE trials


,number,value,datetime_start,datetime_complete,duration,params_reg,params_score_direction,state
12,12,0.914992,2026-05-09 16:39:17.654841,2026-05-09 16:39:17.718835,0 days 00:00:00.063994,3.458561,inverted,COMPLETE
24,24,0.914992,2026-05-09 16:39:18.496833,2026-05-09 16:39:18.550836,0 days 00:00:00.054003,8.120656,inverted,COMPLETE
22,22,0.914992,2026-05-09 16:39:18.377832,2026-05-09 16:39:18.434832,0 days 00:00:00.057000,1.105850,inverted,COMPLETE
21,21,0.914992,2026-05-09 16:39:18.317832,2026-05-09 16:39:18.377832,0 days 00:00:00.060000,1.143952,inverted,COMPLETE
25,25,0.914992,2026-05-09 16:39:18.551833,2026-05-09 16:39:18.615853,0 days 00:00:00.064020,2.159943,inverted,COMPLETE
18,18,0.914992,2026-05-09 16:39:18.125834,2026-05-09 16:39:18.189833,0 days 00:00:00.063999,2.774239,inverted,COMPLETE
17,17,0.914992,2026-05-09 16:39:18.058833,2026-05-09 16:39:18.125834,0 days 00:00:00.067001,7.940731,inverted,COMPLETE
16,16,0.914992,2026-05-09 16:39:17.916834,2026-05-09 16:39:18.057837,0 days 00:00:00.141003,1.234196,inverted,COMPLETE
1,1,0.914992,2026-05-09 16:39:16.939834,2026-05-09 16:39:17.002832,0 days 00:00:00.062998,3.776663,inverted,COMPLETE
14,14,0.914992,2026-05-09 16:39:17.787834,2026-05-09 16:39:17.843835,0 days 00:00:00.056001,5.006492,inverted,COMPLETE


## Финальное обучение на полном train и сравнение по сегментам

In [13]:
all_eligible = gt["Код клиента"].tolist()

best_als = ALSRecommender(**study_als.best_params).fit(tf["interaction_features"])
best_ease = EASECalibratedRecommender(**study_ease.best_params).fit(tf["interaction_features"])

predictions = {
    "ALS tuned": best_als.predict_scores(all_eligible),
    "EASE tuned": best_ease.predict_scores(all_eligible),
}

results_by_segment = evaluate_predictions_by_segments(predictions, segments)
display(results_by_segment)

,Сегмент,N клиентов,Модель,HR@1,P@2,R@2,NDCG@3,MAP@3
0,All,3536,ALS tuned,0.990,0.586,0.989,0.991,0.993
1,All,3536,EASE tuned,0.976,0.557,0.960,0.975,0.973
2,Cold-start,166,EASE tuned,0.994,0.521,0.983,0.991,0.993
3,Cold-start,166,ALS tuned,0.988,0.524,0.986,0.990,0.993
4,Hot+,244,ALS tuned,0.869,0.797,0.906,0.903,0.914
5,Hot+,244,EASE tuned,0.889,0.689,0.786,0.881,0.861
6,Inertial,3134,ALS tuned,0.999,0.573,0.995,0.997,0.999
7,Inertial,3134,EASE tuned,0.981,0.549,0.971,0.981,0.980
8,Potential,402,ALS tuned,0.920,0.687,0.943,0.941,0.948
9,Potential,402,EASE tuned,0.933,0.619,0.869,0.927,0.915


In [14]:
pivot_ndcg = results_by_segment.pivot(index="Сегмент", columns="Модель", values="NDCG@3")
pivot_map = results_by_segment.pivot(index="Сегмент", columns="Модель", values="MAP@3")

print("NDCG@3 по сегментам")
display(pivot_ndcg)

print("MAP@3 по сегментам")
display(pivot_map)

comparison = pivot_ndcg.copy()
"model_cols = comparison.select_dtypes(include=\"number\").columns\n",
"comparison[\"winner\"] = comparison[model_cols].idxmax(axis=1)\n",
"comparison[\"delta_abs\"] = (comparison[model_cols].max(axis=1) - comparison[model_cols].min(axis=1)).round(4)\n",
display(comparison.reset_index())

NDCG@3 по сегментам


Модель,ALS tuned,EASE tuned
Сегмент,,
All,0.991,0.975
Cold-start,0.990,0.991
Hot+,0.903,0.881
Inertial,0.997,0.981
Potential,0.941,0.927


MAP@3 по сегментам


Модель,ALS tuned,EASE tuned
Сегмент,,
All,0.993,0.973
Cold-start,0.993,0.993
Hot+,0.914,0.861
Inertial,0.999,0.980
Potential,0.948,0.915


Модель,Сегмент,ALS tuned,EASE tuned
0,All,0.991,0.975
1,Cold-start,0.990,0.991
2,Hot+,0.903,0.881
3,Inertial,0.997,0.981
4,Potential,0.941,0.927


## Сохранение результатов

In [15]:
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

results_by_segment.to_csv(output_dir / "als_vs_ease_optuna_segment_results.csv", index=False, encoding="utf-8-sig")
best_params_summary.to_csv(output_dir / "als_vs_ease_optuna_best_params.csv", index=False, encoding="utf-8-sig")
als_trials.to_csv(output_dir / "als_optuna_trials.csv", index=False, encoding="utf-8-sig")
ease_trials.to_csv(output_dir / "ease_optuna_trials.csv", index=False, encoding="utf-8-sig")

print("Сохранено:")
print(output_dir / "als_vs_ease_optuna_segment_results.csv")
print(output_dir / "als_vs_ease_optuna_best_params.csv")
print(output_dir / "als_optuna_trials.csv")
print(output_dir / "ease_optuna_trials.csv")

Сохранено:
outputs\als_vs_ease_optuna_segment_results.csv
outputs\als_vs_ease_optuna_best_params.csv
outputs\als_optuna_trials.csv
outputs\ease_optuna_trials.csv
